# ECON2041 Week 6 tutorial: your first regression

### What you'll be able to do by the end

- Match the simple regression model to the variables in our question
- Fit a regression with `ols()` and interpret its intercept and slope
- Rebuild the slope from the covariance and the variance
- Use the fitted line to predict mean reading scores, and spot a prediction that overreaches
- Explain why we say "associated with" rather than "causes"

### Different types of cells

- 🟢 Read and run: these cells have pre-written code, you can simply run them, read and interpret the output
- ✏️ You write: your turn to try to write code or sometimes add a short written answer
- 🤔 You think: stop and think on your own before running the next cell or reading on
- 💬 Discuss: talk with your neighbors before typing or running 
- ⭐ Optional: extra exercise if you have time  Optional: extra exercise if you have time 

## Setup

Our usual setup block plus one new import: `ols` from `statsmodels`, our main regression tool.

In [ ]:
# 🟢 Read and run: our standard ECON2041 setup block
import numpy as np               # numerical tools (nicknamed np)
import pandas as pd              # data tools (nicknamed pd)
import matplotlib.pyplot as plt  # plotting tools (nicknamed plt)
import seaborn as sns            # statistical charts (nicknamed sns)
from statsmodels.formula.api import ols  # ordinary least squares regression

# Keep scalar output plain under NumPy 2 (0.5, not np.float64(0.5)).
if np.lib.NumpyVersion(np.__version__) >= "2.0.0":
    np.set_printoptions(legacy="1.25")

DATA = "https://emiliatjernstrom.com/econ2041/data"   # Unit datasets live at this web address

print("Setup done!")

## The question and the data

The week 5 Essential Concepts recordings introduced simple regression: a straight line, $\hat{y} = \hat{\beta}_0 + \hat{\beta}_1 x$, chosen by least squares.
In the week 5 live lecture we fit that line on real data: math scores against family background, for a random sample of 300 Australian students in PISA 2022.

Today we fit the same kind of line to a different outcome, reading scores, using all 12,136 complete records in our teaching file.

> How much higher do students from more advantaged families score in reading, on average?

Source: [OECD PISA](https://www.oecd.org/en/about/programmes/pisa.html), via the [learningtower project](https://kevinwang09.github.io/learningtower/).

We'll focus on two variables today:

- `read`: the student's PISA reading score
- `escs`: the index of economic, social and cultural status, our "socio-economic family background" variable

In [ ]:
# 🟢 Read and run: load pisa-australia-2022.csv into a dataframe called pisa
pisa = pd.read_csv(f"{DATA}/pisa-australia-2022.csv")

pisa.head()

## See the relationship

In the week 5 live lecture we studied the scatterplot of math scores against `escs` for 300 students.
Now you draw the reading version, for all 12,136 records.

### Question 1: how do reading scores vary with family background?

In [ ]:
# ✏️ You write: a scatterplot of read (vertical) against escs (horizontal), one dot per student
# Hint: sns.scatterplot with data=pisa, x and y,
#       plus s=8 and alpha=0.2 (alpha adds transparency to make it easier to see with many observations)

🤔 Look before you compute:

- As you move right, toward students with a higher `escs`, do the dots drift up, down, or neither?
- Pick any narrow range of `escs` and scan up and down: how much do reading scores vary among students with nearly the same family background?

## Name and fit the simple regression model

For each student, the observed reading score equals the mean score at that family-background value plus an individual gap. The week 5 Essential Concepts recordings wrote that idea as:

$$y_i = \beta_0 + \beta_1 x_i + u_i$$

| Model part | Common names | Variable today |
|---|---|---|
| $y_i$ | left-hand-side variable; outcome; dependent variable | `read` |
| $x_i$ | right-hand-side variable; explanatory variable; independent variable | `escs` |
| $\beta_0$ | intercept | the mean relationship's height at `escs` $= 0$ |
| $\beta_1$ | slope; coefficient on $x_i$ | the change in mean reading score for a 1-point difference in `escs` |
| $u_i$ | error term | the part of student $i$'s score not captured by the population line |

"Independent variable" is a traditional name. It does not mean that `escs` is statistically independent of `read`, and none of these names makes the relationship causal. We will usually say "explanatory variable".

### Question 2: what line does least squares choose?

In the `ols()` formula, the outcome goes to the left of `~` and the explanatory variable goes to the right: `read ~ escs`.
`model.params` reports the estimated intercept first and the estimated slope second.

In [ ]:
# 🟢 Read and run: fit read on escs with ordinary least squares
model = ols("read ~ escs", data=pisa).fit()

model.params.round(1)

### Question 3: what does the fitted line say?

🤔 Write out the fitted line in the live lecture's notation, $\widehat{\text{read}} = \hat{\beta}_0 + \hat{\beta}_1 \times \text{escs}$, with the two estimates filled in.
What does each number mean here?

For scale, compare the slope with the standard deviation of reading scores, `pisa["read"].std()`.

🤔 Vocabulary check:

- Which symbols in the simple model are population parameters?
- What is the estimator?
- Which two numbers are the estimates in this teaching file?
- Why would another sample give different estimates?

### Question 4: can you rebuild the slope from the covariance and the variance?

The live lecture derived the slope formula from the least-squares rule:

$$\hat{\beta}_1 = \frac{\text{cov}(X, Y)}{\text{var}(X)}$$

Both ingredients live in the covariance table you met in the week 5 tutorial: the off-diagonal entry is the covariance, and the diagonal entries are the two variances.

In [ ]:
# ✏️ You write: create a table showing the covariances between reading scores and socioeconomic status,
# using pisa[[...]].cov(), and call the table cov_table

In [ ]:
# ✏️ You write: rebuild the slope by dividing the covariance of read and escs by the variance of escs
# Hint: Use cov_table.loc[row_name, column_name] to select each value
# Variances are on the diagonal of the table

### ⭐ Optional extra (if time): can you rebuild the intercept too?

Once OLS finds the slope, it shifts the line up or down until it passes through the point formed by the two sample means, $(\overline{\text{escs}}, \overline{\text{read}})$.
That fact pins down the intercept:

$$\hat{\beta}_0 = \overline{\text{read}} - \text{slope} \times \overline{\text{escs}}$$

In [ ]:
# ✏️ You write (optional): rebuild the intercept from the two sample means and the slope you just computed
# Hint: pisa["read"].mean() and pisa["escs"].mean() return the two means

## Draw and interpret the fitted line

We have the two numbers; now we look at the line they describe.
`sns.regplot` draws the scatterplot and the least-squares line in one command.

### Question 5: where does the fitted line sit in the data?

The code below draws the scatterplot with the least-squares line through it, and marks `escs` $= 0$ with a dashed vertical line; question 6 comes back to why 0 is worth marking.
Only two blanks are yours to fill.

In [ ]:
# ✏️ You fill in: the code below is complete except for the two blanks.
# Fill them in, then remove the leading # from each line and run
# (in Colab: select the lines and press Ctrl+/ to uncomment them all at once).

# sns.regplot(data=pisa,                            # same data as question 1
#             x=________,                           # the explanatory variable, in quotes
#             y=________,                           # the outcome, in quotes
#             ci=None,                              # skip the shaded band around the line for now
#             scatter_kws={"s": 8, "alpha": 0.2,    # small see-through dots, like question 1
#                          "color": "purple"},      # same purple as question 1
#             line_kws={"color": "orange"})         # draw the fitted line in orange
# plt.axvline(0, color="gray", linestyle="--")      # dashed vertical line at escs = 0
# plt.xlabel("Economic, social and cultural status (escs)")
# plt.ylabel("Reading score")
# plt.show()

🤔 The live lecture built its line by taking the mean math score inside narrow slices of `escs`.

Where is the conditional mean, $E[Y \mid X]$, in this picture?

## Predict with the fitted line

A fitted line turns any value of `escs` into an estimated mean reading score: plug the value into the equation and read the result.
The next two questions walk through what those predictions mean, and where they stop being trustworthy.

### Question 6: what does the line predict at `escs` = 1?

In [ ]:
# ✏️ You write: use the fitted coefficients to predict the mean reading score at escs = 1
escs_value = 1
# Complete these two lines, then run the cell:
# prediction_escs_1 = ________
# prediction_escs_1
# Hint: model.params["Intercept"] returns the intercept.
#       The slope lives in model.params too, stored under its variable's name.

🤔 At `escs` $= 0$, which term in the fitted equation remains? What does that number mean here?

### Question 7: what happens across a range of predictions?

Question 6 predicted at a single `escs` value
- `model.predict()` is a way of handing the model several `escs` values at once and asking for a prediction at each
- the code below hands it four values, collected under the name `new_students` 
- the fitted line returns a prediction for any `escs` value we hand it

🤔 Before you run the code below: use the fitted line, $\widehat{\text{read}} = 484.4 + 41.4 \times \text{escs}$, to work out the prediction at each of the four `escs` values ($-1, 0, 1, 2$) by hand.

In [ ]:
# 🟢 Read and run: predictions at four values of escs
new_students = pd.DataFrame({"escs": [-1, 0, 1, 2]})
new_students["predicted_read"] = model.predict(new_students)
new_students.round(1)

🤔 Read the four predictions against the fitted equation:

- How close were your by-hand numbers? If they are off by a little, where does the gap come from?
- How much does the prediction change from each `escs` value to the next? Why that number?
- Nothing stops you from plugging in `escs` $= 5$. How much would you trust that prediction, and why?

## What we can and can't learn from the regression

The slope says that students whose family background index is 1 point higher score about 41 points higher in reading, on average.

### Question 8: what does that sentence let us claim?

💬 With your neighbors:

- Does the slope mean that *raising* a family's `escs`, say through higher income or more books in the house, would raise their child's reading score by about 41 points?
- Name two other stories that could produce this association even if family background itself had no effect on reading scores